# Sentiment Analysis - Colab Training
Trains DistilBERT on Twitter US Airline Sentiment with free GPU.
After training → downloads model.zip → extract to `models/sentiment_model/` locally.

In [ ]:
!pip install -q torch transformers datasets accelerate evaluate scikit-learn pandas numpy

In [ ]:
import re, os, json, zipfile, shutil
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import Dataset, load_dataset

In [ ]:
DATA_SOURCE = 'osanseviero/twitter-airline-sentiment'
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 2e-5

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
print('Loading dataset...')
ds = load_dataset(DATA_SOURCE, split='train')
df = ds.to_pandas()[['text', 'airline_sentiment']]
print(f'Rows: {len(df)}')

In [ ]:
def clean_text(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['clean_text'] = df['text'].astype(str).apply(clean_text)
df['label'] = df['airline_sentiment'].map(label_map).astype(int)
df = df.dropna(subset=['label']).reset_index(drop=True)

print(df['airline_sentiment'].value_counts())

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['clean_text'].tolist(), df['label'].tolist(),
    test_size=0.2, random_state=42, stratify=df['label']
)
print(f'Train: {len(train_texts)}, Val: {len(val_texts)}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tok(texts):
    return tokenizer(texts, max_length=MAX_LEN, truncation=True, padding='max_length')

train_ds = Dataset.from_dict({'input_ids': tok(train_texts)['input_ids'], 'attention_mask': tok(train_texts)['attention_mask'], 'labels': train_labels})
val_ds = Dataset.from_dict({'input_ids': tok(val_texts)['input_ids'], 'attention_mask': tok(val_texts)['attention_mask'], 'labels': val_labels})

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label={0:'negative',1:'neutral',2:'positive'},
    label2id={'negative':0,'neutral':1,'positive':2}
).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1, 'precision': p, 'recall': r}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=True,
    report_to='none',
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(1)],
)

trainer.train()

In [ ]:
results = trainer.evaluate()
for k, v in results.items():
    print(f'{k}: {v:.4f}')

In [ ]:
model.save_pretrained('/content/sentiment_model')
tokenizer.save_pretrained('/content/sentiment_model')
print('Model saved to /content/sentiment_model/')

In [ ]:
zip_path = '/content/sentiment_model.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk('/content/sentiment_model'):
        for file in files:
            zf.write(os.path.join(root, file),
                     os.path.relpath(os.path.join(root, file), '/content/sentiment_model'))

from google.colab import files
files.download(zip_path)
print('Download started. Extract to models/sentiment_model/ on your local machine.')